# Code Overlay Worktree Epoch Validation Lease

**Status:** Design written; implementation requires review.

**Revision epic:** `bd-hxut`  
**Supersedes:** the synchronous fsmonitor-token assumption in `bd-2r17`  
**Architecture solve:** `sol_c3dcb70186b24f3a`  
**Mismatch-rate bound:** `sol_72bac8d1a8324f21`

## Executive decision

For a dedicated SPUR worktree with one registered writer, serialized mutations, a healthy filesystem watcher, and a completed exact baseline, validate `code_*` responses with a per-worktree generation epoch. A request reads epoch `E1`, pins immutable overlay generation `E1`, executes entirely against that generation, then reads `E2`. It may return only when the worktree is stable and `E1 = generation_epoch = E2`.

Every uncertain case—shared main roots, multiple or unknown writers, watcher lag or overflow, startup before reconciliation, branch/index changes, and retry exhaustion—uses the existing exact Git observation path. The epoch is an optimization lease, never an independent source of truth.

**Native NS-Mermaid profiles pinned:** `relational_lia@1`, `state_invariant_lia@1`, and `sequence_trace@1`. The architecture registry profile is unavailable, so the architecture view below is explicitly conceptual; the adjacent native cells carry the executable contracts.

## Problem and measured evidence

The immutable overlay-generation architecture is already doing its job. A warm symbol query against a pinned generation is measured in microseconds, while end-to-end MCP latency is dominated by two exact Git observations around the query.

| Stage | Fresh measurement | Interpretation |
|---|---:|---|
| Pinned generation query | 0.0004–0.0022 ms p50 | merge/filter/sort/dedup are not repeated on the warm path |
| Exact Git observation A | 33.5–33.8 ms p50 | pre-query source-of-truth fence |
| Exact Git observation B | 33.5–33.8 ms p50 | post-query source-of-truth fence |
| Warm full MCP | 82.4–84.7 ms p50 | two Git observations dominate |
| Warm rebuild/base-load/finalization | 0 observed | generation cache and pinning are effective |

`overlay_response_for_backend` currently prepares an exact overlay snapshot, obtains or builds an `OverlayGeneration`, pins a `PinnedGenerationClient`, executes the handler, and performs `authoritative_overlay_identity` again before committing. The change proposed here replaces those request-time exact A/B observations only for an eligible owned worktree.

The persisted mismatch-rate solve is deliberately conservative: an 85 ms exposure window stays below a 0.1% mismatch probability when the mean relevant mutation interval is at least 85 seconds, below 1% at 8.5 seconds, and below 5% at 1.7 seconds. Probability is supporting evidence, not the correctness mechanism; the epoch fence is.

## Goals, non-goals, and terms

### Goals

- Preserve the working tree as the source of truth for every returned `code_*` result.
- Remove both exact Git observations from eligible, unchanged, warm requests.
- Make mutation invalidation global to a worktree, with a changed-path journal for incremental rebuilds.
- Keep the existing identity-keyed generation cache, singleflight build, compatible seed, and atomic publish behavior.
- Fail closed to exact Git validation whenever the epoch lease cannot prove eligibility.
- Target dedicated-worktree warm end-to-end p95 below 10 ms with zero request-time exact observations.

### Non-goals

- An mtime-only cache key or TTL as a freshness authority.
- Assuming Git's public fsmonitor API provides synchronous per-request observation tokens.
- Eliminating the exact Git implementation.
- Making shared main workspaces or arbitrary external writers use the fast path.
- Choosing a periodic reconciliation interval by intuition; any interval is measurement- and solve-owned.

### Terms

- **Worktree epoch:** a monotonically increasing process-visible generation for one canonical worktree root.
- **Stable epoch:** an even epoch whose published immutable overlay generation represents every registered mutation up to that epoch.
- **Dirty epoch:** an odd epoch entered before mutation and left only after changed paths are rebuilt and published.
- **Exact baseline:** a successful exact Git observation used at startup or after uncertainty to re-establish source-of-truth state.
- **Ownership lease:** proof that the root is a dedicated SPUR worktree with one registered mutation coordinator.

## Validation-route eligibility

The epoch lease is available only when all five safety predicates agree:

1. The canonical root has an active SPUR ownership lease.
2. All agent writes in that worktree are serialized through the registered mutation coordinator.
3. The filesystem watcher is healthy and has not overflowed or lost continuity.
4. An exact baseline has completed since process start, ownership change, or watcher recovery.
5. The root is not classified as shared.

If any predicate is false or unknown, the route is `exact_fallback`. This classification is evaluated before the request begins and can be revoked asynchronously by the watcher or ownership manager. The epoch route cannot be enabled by configuration alone.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-EPOCH-ELIGIBILITY
@type ValidationRoute = enum[epoch_lease, exact_fallback]
@input owned_worktree: Bool
@input serialized_writes: Bool
@input watcher_healthy: Bool
@input exact_baseline_valid: Bool
@input shared_root: Bool
@output status: ValidationRoute
@requires PRE: true`"]

    EPOCH["`@branch EPOCH
@when owned_worktree and serialized_writes and watcher_healthy and exact_baseline_valid and not shared_root
@ensures EPOCH_ROUTE: status = epoch_lease`"]

    EXACT["`@branch EXACT
@when not (owned_worktree and serialized_writes and watcher_healthy and exact_baseline_valid and not shared_root)
@ensures EXACT_ROUTE: status = exact_fallback`"]

    CHECK["`@verify ELIGIBILITY_DETERMINISTIC: prove determinism
@verify ELIGIBILITY_COVERAGE: prove partition_coverage
@verify ELIGIBILITY_EXCLUSIVE: prove partition_exclusive
@verify ELIGIBILITY_STATUSES: witness each status
@verify EPOCH_REACHABLE: witness branch EPOCH
@verify EXACT_REACHABLE: witness branch EXACT`"]

    SPEC --> EPOCH --> CHECK
    SPEC --> EXACT --> CHECK

## Target architecture and ownership

```mermaid
flowchart LR
  subgraph Control[Worktree control plane]
    Owner[Ownership lease]
    Mutator[Mutation coordinator]
    Epoch[Epoch state<br/>stable or dirty]
    Journal[Changed-path journal]
    Watcher[Filesystem watcher]
  end

  subgraph Data[Overlay data plane]
    Builder[Incremental generation builder]
    Cache[Identity-keyed cache<br/>singleflight]
    Pinned[Pinned query client]
  end

  subgraph Validation[Response validation]
    Gate[Epoch response gate]
    Exact[Exact Git fallback]
  end

  Owner --> Mutator
  Mutator -->|begin and finish| Epoch
  Mutator -->|changed paths| Journal
  Watcher -->|external change or overflow| Epoch
  Journal --> Builder --> Cache --> Pinned
  Epoch -->|E1| Pinned
  Pinned --> Gate
  Epoch -->|E2| Gate
  Gate -->|stable and equal| Result[Return result]
  Gate -->|mismatch or uncertainty| Exact --> Result
```

### Component responsibilities

- `WorktreeOwnershipLease` canonicalizes the root and distinguishes dedicated SPUR worktrees from shared or unknown roots.
- `MutationCoordinator` is the only fast-path writer. It transitions the epoch to dirty before the first filesystem mutation and does not publish stable until the generation is ready.
- `WorktreeEpochState` owns the monotonic epoch, stable/dirty state, watcher health, and exact-baseline validity.
- `ChangedPathJournal` records create, modify, delete, and both sides of rename operations. It feeds the existing incremental generation constructor.
- `OverlayGeneration` remains immutable. The existing cache and singleflight publication mechanisms remain unchanged.
- `ExactReconciler` owns startup, shared-root, watcher-loss, branch/index-change, and retry-exhaustion recovery.

The epoch is global per canonical worktree, not per returned file. A change outside the current query can alter dependency closure, name resolution, shadowing, or path visibility, so local file mtimes cannot prove response safety.

In [ ]:
stateDiagram-v2
    [*] --> Stable
    Stable --> Dirty: begin_write
    Dirty --> Stable: publish_generation

    note right of Stable
      @spec CODE-OVERLAY-EPOCH-LIFECYCLE
      @type EpochEvent = enum[begin_write, publish_generation]
      @input event: EpochEvent
      @state-var epoch: Int
      @state-var published_epoch: Int
      @state-var stable: Bool
      @requires PRE: epoch = 0 and published_epoch = 0 and stable = true
      @state Stable
      @invariant ORDERED: published_epoch <= epoch
      @invariant STABLE_MATCH: not stable or published_epoch = epoch
      @invariant DIRTY_AHEAD: stable or published_epoch < epoch
      @transition PUBLISH_GENERATION
      @from Dirty
      @to Stable
      @event event = publish_generation
      @guard stable = false and published_epoch < epoch
      @update epoch' = epoch + 1
      @update published_epoch' = epoch + 1
      @update stable' = true
      @verify ORDERED_INIT: prove initiate ORDERED
      @verify STABLE_MATCH_INIT: prove initiate STABLE_MATCH
      @verify DIRTY_AHEAD_INIT: prove initiate DIRTY_AHEAD
      @verify ORDERED_BEGIN: prove preserve ORDERED on BEGIN_WRITE
      @verify ORDERED_PUBLISH: prove preserve ORDERED on PUBLISH_GENERATION
      @verify STABLE_MATCH_BEGIN: prove preserve STABLE_MATCH on BEGIN_WRITE
      @verify STABLE_MATCH_PUBLISH: prove preserve STABLE_MATCH on PUBLISH_GENERATION
      @verify DIRTY_AHEAD_BEGIN: prove preserve DIRTY_AHEAD on BEGIN_WRITE
      @verify DIRTY_AHEAD_PUBLISH: prove preserve DIRTY_AHEAD on PUBLISH_GENERATION
    end note

    note right of Dirty
      @state Dirty
      @transition BEGIN_WRITE
      @from Stable
      @to Dirty
      @event event = begin_write
      @guard stable = true and published_epoch = epoch
      @update epoch' = epoch + 1
      @update published_epoch' = published_epoch
      @update stable' = false
    end note

## Healthy-path protocol

Stable epochs are even; dirty epochs are odd. The stable/dirty bit is authoritative, while parity is a cheap diagnostic convention.

1. Before any registered write, atomically increment epoch `E` to the next odd value and mark the worktree dirty.
2. Apply filesystem mutations while appending normalized create/modify/delete/rename paths to the journal.
3. Build the next immutable overlay generation from the prior compatible generation plus the changed-path dependency closure.
4. Atomically publish the generation, increment epoch once more to the next even value, set `published_epoch = epoch`, and mark stable.
5. A request reads stable `E1`, obtains the generation for `E1`, and pins it for the full handler execution.
6. Immediately before returning, it reads `E2`. It commits only when stable and `E1 = generation_epoch = E2`.
7. A first mismatch retries once against the newest stable generation. A second mismatch or any uncertainty invokes exact fallback.

No Git subprocess or repository walk is required on the eligible unchanged path. Registered edits cannot race a committed response because the pre-write dirty transition happens before bytes change and the response gate observes a changed epoch or dirty state.

In [ ]:
sequenceDiagram
    participant Agent
    participant Mutator
    participant Epoch
    participant Builder
    participant Query
    participant Gate

    Note over Agent,Gate: @spec CODE-OVERLAY-EPOCH-REQUEST-PROTOCOL<br/>@input start_epoch: Int<br/>@input generation_epoch: Int<br/>@input end_epoch: Int<br/>@requires HEALTHY_TRACE: start_epoch >= 0 and start_epoch = generation_epoch and generation_epoch = end_epoch

    Agent->>Mutator: begin registered write
    Note over Agent,Mutator: @message BEGIN_WRITE<br/>@from Agent<br/>@to Mutator<br/>@event begin registered write<br/>@order 1<br/>@when true<br/>@ensures BEGIN_RECORDED: true

    Mutator->>Epoch: mark dirty and increment
    Note over Mutator,Epoch: @message MARK_DIRTY<br/>@from Mutator<br/>@to Epoch<br/>@event mark dirty and increment<br/>@order 2<br/>@when true<br/>@ensures DIRTY_RECORDED: true

    Agent->>Mutator: finish registered write
    Note over Agent,Mutator: @message FINISH_WRITE<br/>@from Agent<br/>@to Mutator<br/>@event finish registered write<br/>@order 3<br/>@when true<br/>@ensures FINISH_RECORDED: true

    Mutator->>Builder: publish immutable generation
    Note over Mutator,Builder: @message PUBLISH_GENERATION<br/>@from Mutator<br/>@to Builder<br/>@event publish immutable generation<br/>@order 4<br/>@when true<br/>@ensures GENERATION_MATCHES: generation_epoch = start_epoch

    Builder->>Epoch: mark stable and increment
    Note over Builder,Epoch: @message MARK_STABLE<br/>@from Builder<br/>@to Epoch<br/>@event mark stable and increment<br/>@order 5<br/>@when generation_epoch = start_epoch<br/>@ensures STABLE_RECORDED: true

    Agent->>Epoch: acquire stable epoch
    Note over Agent,Epoch: @message ACQUIRE_EPOCH<br/>@from Agent<br/>@to Epoch<br/>@event acquire stable epoch<br/>@order 6<br/>@when true<br/>@ensures START_MATCHES_GENERATION: start_epoch = generation_epoch

    Epoch->>Query: pin immutable generation
    Note over Epoch,Query: @message PIN_GENERATION<br/>@from Epoch<br/>@to Query<br/>@event pin immutable generation<br/>@order 7<br/>@when start_epoch = generation_epoch<br/>@ensures PINNED: generation_epoch = start_epoch

    Agent->>Query: execute code query
    Note over Agent,Query: @message CODE_QUERY<br/>@from Agent<br/>@to Query<br/>@event execute code query<br/>@order 8<br/>@when start_epoch = generation_epoch<br/>@ensures QUERY_PINNED: true

    Query->>Epoch: validate response epoch
    Note over Query,Epoch: @message VALIDATE_EPOCH<br/>@from Query<br/>@to Epoch<br/>@event validate response epoch<br/>@order 9<br/>@when true<br/>@ensures END_MATCHES_GENERATION: end_epoch = generation_epoch

    Gate-->>Agent: return exact-generation result
    Note over Agent,Gate: @message RETURN_RESULT<br/>@from Gate<br/>@to Agent<br/>@event return exact-generation result<br/>@order 10<br/>@when start_epoch = generation_epoch and generation_epoch = end_epoch<br/>@ensures RESPONSE_SAFE: start_epoch = end_epoch

    Note over Agent,Gate: @verify EPOCH_REQUEST_TRACE: prove sequence_protocol

## Response commit gate

The response gate is the only place allowed to turn an epoch lease into a user-visible result.

| Condition | Action |
|---|---|
| Epoch route, stable, and `E1 = generation_epoch = E2` | Commit pinned result |
| Epoch route, first mismatch or dirty observation | Discard result and retry once |
| Epoch route, repeated mismatch | Exact Git fallback |
| Exact route selected before query | Exact Git validation |
| Exact validation succeeds | Commit exact result |
| Exact validation fails | Return an error; never return the speculative result |

A response does not become less correct because the graph index is stale: the overlay remains the source-of-truth correction layer. This lease only changes how the service proves the overlay generation did not change during the request. It never authorizes returning a generation whose epoch differs from the worktree.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-EPOCH-RESPONSE-GATE
@type CommitAction = enum[commit_epoch, retry, commit_exact, error]
@input stable: Bool
@input start_epoch: Int
@input generation_epoch: Int
@input end_epoch: Int
@input first_attempt: Bool
@input exact_route: Bool
@input exact_ok: Bool
@output status: CommitAction
@requires NONNEGATIVE_EPOCHS: start_epoch >= 0 and generation_epoch >= 0 and end_epoch >= 0`"]

    COMMIT["`@branch COMMIT
@when not exact_route and stable and start_epoch = generation_epoch and generation_epoch = end_epoch
@ensures COMMIT_STATUS: status = commit_epoch`"]

    RETRY["`@branch RETRY
@when not exact_route and (not stable or start_epoch != generation_epoch or generation_epoch != end_epoch) and first_attempt
@ensures RETRY_STATUS: status = retry`"]

    EXACT["`@branch EXACT
@when (exact_route or (not exact_route and (not stable or start_epoch != generation_epoch or generation_epoch != end_epoch) and not first_attempt)) and exact_ok
@ensures EXACT_STATUS: status = commit_exact`"]

    ERROR["`@branch ERROR
@when (exact_route or (not exact_route and (not stable or start_epoch != generation_epoch or generation_epoch != end_epoch) and not first_attempt)) and not exact_ok
@ensures ERROR_STATUS: status = error`"]

    CHECK["`@verify GATE_DETERMINISTIC: prove determinism
@verify GATE_COVERAGE: prove partition_coverage
@verify GATE_EXCLUSIVE: prove partition_exclusive
@verify GATE_STATUSES: witness each status
@verify COMMIT_REACHABLE: witness branch COMMIT
@verify RETRY_REACHABLE: witness branch RETRY
@verify EXACT_REACHABLE: witness branch EXACT
@verify ERROR_REACHABLE: witness branch ERROR`"]

    SPEC --> COMMIT --> CHECK
    SPEC --> RETRY --> CHECK
    SPEC --> EXACT --> CHECK
    SPEC --> ERROR --> CHECK

## Cache, invalidation, and concurrency semantics

- **No TTL freshness rule.** Cache entries remain valid while their immutable identity and epoch remain addressable; capacity eviction is an operational concern, not a correctness deadline.
- **Existing cache preserved.** Identity-keyed lookup, compatible seeding, singleflight construction, and atomic publication continue to own `OverlayGeneration` reuse.
- **Epoch-keyed publication.** A stable generation records the canonical worktree, exact-baseline identity, and published epoch. Requests never attach a newer epoch to older data.
- **Incremental rebuild.** The changed-path journal drives rebuild of affected file chunks, symbol segments, visibility indexes, and dependency-closure adjacency; unchanged segments retain structural sharing.
- **Serialized registered writes.** The mutation coordinator serializes begin/apply/publish for one owned worktree. Queries do not hold the mutation lock; they rely on two atomic epoch reads and immutable pinning.
- **External changes.** A watcher event revokes stable eligibility before rebuild. Overflow, discontinuity, or unknown path semantics revoke the lease and require exact reconciliation.
- **Global invalidation.** New, deleted, renamed, ignored, staged, branch, and index changes affect the worktree epoch even when the current query does not mention the changed file.
- **Memory ordering.** Publish the immutable generation before the release-store of the stable epoch; requests acquire-load epoch before pinning and before committing.

## Failure and fallback matrix

| Situation | Required behavior |
|---|---|
| Dedicated SPUR worktree; registered writer; watcher healthy | Epoch lease |
| No changes between E1 and E2 | Commit pinned generation |
| Registered write starts during query | Dirty/changed epoch; discard and retry |
| Retry also observes a change | Exact fallback |
| Shared main repository or a second terminal | Exact route from request start |
| Unknown ownership or unregistered writer | Revoke lease; exact reconciliation |
| Watcher reports create/modify/delete/rename | Dirty epoch; journal paths; rebuild |
| Watcher lag, overflow, disconnect, or unsupported filesystem | Revoke lease; exact fallback |
| Process startup or restart | Exact baseline before eligibility |
| HEAD, index, ignore rules, submodule, or sparse-checkout changes | Dirty/reconcile; never mtime-only |
| Generation build fails | Remain dirty; exact fallback or explicit error |
| Exact observation fails | Error; do not return speculative data |

The low expected `A != B` rate justifies optimizing the common route, but it does not justify silently accepting a mismatch. The design makes a mismatch cheap to detect and expensive only to recover.

## Release gate

The optimization stays probe-only until correctness and performance pass together.

- Race tests cover registered writes before pin, during handler execution, and before commit.
- Shared-root, second-writer, watcher-overflow, restart, exact-fallback, create/delete/rename, HEAD/index, and build-failure tests pass.
- Cross-project oracle comparison reports zero result mismatches.
- A 30-run matrix on small, medium, and large repositories reports dedicated unchanged warm p95 below 10 ms.
- Diagnostics confirm zero request-time exact Git observations on the dedicated eligible path.
- Exact fallback remains available and measured.

The 10 ms threshold is a release target derived from the already measured microsecond generation query plus local MCP overhead. The solver's `warm_validation_cost_rank = 1` is ordinal, not a claim that validation already measures 1 ms.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-EPOCH-RELEASE
@type ReleaseDecision = enum[release, probe_only]
@input race_tests_pass: Bool
@input shared_root_tests_pass: Bool
@input watcher_tests_pass: Bool
@input fallback_tests_pass: Bool
@input restart_tests_pass: Bool
@input correctness_mismatches: Int
@input dedicated_warm_p95_us: Int
@input dedicated_exact_observations: Int
@output status: ReleaseDecision
@requires NONNEGATIVE_METRICS: correctness_mismatches >= 0 and dedicated_warm_p95_us >= 0 and dedicated_exact_observations >= 0`"]

    RELEASE["`@branch RELEASE
@when race_tests_pass and shared_root_tests_pass and watcher_tests_pass and fallback_tests_pass and restart_tests_pass and correctness_mismatches = 0 and dedicated_warm_p95_us < 10000 and dedicated_exact_observations = 0
@ensures RELEASE_STATUS: status = release`"]

    PROBE["`@branch PROBE
@when not (race_tests_pass and shared_root_tests_pass and watcher_tests_pass and fallback_tests_pass and restart_tests_pass and correctness_mismatches = 0 and dedicated_warm_p95_us < 10000 and dedicated_exact_observations = 0)
@ensures PROBE_STATUS: status = probe_only`"]

    CHECK["`@verify RELEASE_DETERMINISTIC: prove determinism
@verify RELEASE_COVERAGE: prove partition_coverage
@verify RELEASE_EXCLUSIVE: prove partition_exclusive
@verify RELEASE_STATUSES: witness each status`"]

    SPEC --> RELEASE --> CHECK
    SPEC --> PROBE --> CHECK

## TDD and PRE → POST evaluation contract

Implementation must use strict RED → GREEN and record a solver-backed PRE and POST evaluation for each task.

### Required RED contracts

- A registered write changes the epoch before bytes and prevents a stale response commit.
- Publish occurs only after the matching immutable generation is available.
- Create, delete, rename, ignore, HEAD, and index changes invalidate the worktree globally.
- Shared/unknown roots and a simulated second writer never select the epoch route.
- Watcher overflow, disconnect, and startup without baseline select exact fallback.
- First epoch mismatch retries; second mismatch invokes exact fallback.
- Exact failure never returns the speculative epoch result.
- Unchanged segments preserve structural sharing and exact-oracle equality.

### Performance evaluation

Capture PRE and POST stage timings for at least 30 runs on three project sizes: route selection, E1 read, generation lookup/build, pinned query, E2 read, retry/fallback, serialization, and total MCP latency. Report p50/p95, exact-observation count, generation-build count, mismatch count, and oracle mismatch count.

No release decision may substitute a benchmark for correctness or a proof for measurement.

## Migration and rollout

1. **Observe:** add ownership, epoch, watcher-health, route, mismatch, retry, and exact-fallback diagnostics while exact A/B remains authoritative.
2. **Probe:** enable epoch validation only for dedicated SPUR worktrees; execute the epoch decision but compare against exact validation without affecting returned results.
3. **Release:** permit epoch commit only after the formal release gate and cross-project benchmark pass.
4. **Retain fallback:** keep exact validation available for shared roots, recovery, debugging, and rollback.

Rollback is configuration-only: disable epoch commit and route every request through exact validation. Cache/generation data need not be deleted because immutable identities remain valid historical values.

A periodic exact audit may be added later as defense in depth, but its interval must be selected from measured watcher-loss rates and a bounded risk objective. TTL is not part of the correctness contract.

## Implementation boundaries and dependency DAG

This specification intentionally stops before file-level task assignment. The implementation plan must re-ground symbols against the then-current graph.

1. **Ownership and epoch contract** — canonical worktree ownership, stable/dirty epoch state, memory ordering, diagnostics.
2. **Registered mutation lifecycle** — begin-write barrier, changed-path journal, incremental rebuild, atomic stable publication. Depends on 1.
3. **Watcher and exact reconciliation** — external-change revocation, overflow handling, startup baseline, shared-root classification. Depends on 1.
4. **MCP request lease** — eligibility selection, E1 pin, E2 gate, one retry, exact fallback; preserve existing generation cache. Depends on 2 and 3.
5. **Oracle and performance gates** — race matrix, cross-project correctness, 30-run latency evaluation, release decision. Depends on 4.

Expected seams span worktree ownership/mutation coordination and `spur-graph` MCP validation. Cross-crate scope is permitted only where those responsibilities already live; the plan must not move generation-query logic out of `spur-graph`.

## Formal proof and solver evidence

All native contracts were executed from the sources stored in this notebook. Every generated obligation matched its expected solver status and every proof is fresh.

| Contract | Profile | Obligations | Result | Source hash | Report hash |
|---|---|---:|---|---|---|
| `CODE-OVERLAY-EPOCH-ELIGIBILITY` | `relational_lia@1` | 7 / 7 | verified | `b6cda53209fb2a072a730104200a2c1299312f6a7b55a0aae991f6e9d7e32443` | `a87fd70b436b934a967b18e4a4cdb2791fdb539ad135f24839037f372893041a` |
| `CODE-OVERLAY-EPOCH-LIFECYCLE` | `state_invariant_lia@1` | 18 / 18 | verified | `d796c4d48cb32c6a605fd139e1a3685817bc3de905e7e8112c17f72587365982` | `fe09f2af0d8396c1d49d43d2d5235794676146fd4a601bdb399207910ce647de` |
| `CODE-OVERLAY-EPOCH-REQUEST-PROTOCOL` | `sequence_trace@1` | 1 / 1 | verified | `59e2f59aeacef726458903cab9b4c7531962d04e16b72e73463295e36514988d` | `2a094b1b6ea0056a7c46c6a2574d9a60c9b902a2711d6d141c0c7b24f2da2ef6` |
| `CODE-OVERLAY-EPOCH-RESPONSE-GATE` | `relational_lia@1` | 11 / 11 | verified | `bc9059f9a4e088bf403ae3f3e2ec7300561d1758b712905b67eb4930de4bb035` | `84ca489528af4a37a9938658ee7d06d645c7f7ccdbf84064cb1ed03dd90f3dca` |
| `CODE-OVERLAY-EPOCH-RELEASE` | `relational_lia@1` | 5 / 5 | verified | `d4a4c7d6fbdb85b8220a84f08c219637f98806b4e1ca154775da57516276f2c7` | `43d00bfa1827695e3e03d5d42b4a6053f06e6f55890dae1944db5ef241100bc8` |

### Persisted optimization

- Architecture solve: `sol_c3dcb70186b24f3a`
- Solver: Z3 4.16.0
- Status: SAT
- Optimization: lexicographic minimize, complete termination
- Selected model: `choose_epoch_fallback = true`, `choose_exact_ab = false`, `choose_mtime_only = false`
- Required envelope: owned worktree, serialized writes, healthy watcher, exact fallback available
- Objective: `warm_validation_cost_rank = 1` (ordinal target; the exact A/B route is ranked 67 from measured validation cost)

The mismatch-rate bound `sol_72bac8d1a8324f21` is SAT and gives conservative mean-mutation intervals of 85 s for 0.1%, 8.5 s for 1%, 1.7 s for 5%, and 0.85 s for 10% over an 85 ms exposure window. Correctness still comes from the epoch protocol and exact fallback, not from this probability model.